# Finance Lab 1: Financial Analysis Agent -- Market Data, Risk & ESG

**Series**: Agentic AI Science Playbook -- Finance Domain  
**Goal**: Build an agent that analyzes market data, assesses portfolio risk, and screens for ESG compliance.  
**Prerequisites**: Foundation Labs 0-4.  
**Time**: ~60 min.

---

## Background: Quantitative Finance & AI Agents

Quantitative finance sits at the intersection of mathematics, statistics, and computer science. From pricing derivatives to managing trillion-dollar portfolios, the field has always been computationally intensive. Today, AI agents are transforming how financial professionals interact with data:

- **Market analysis** traditionally requires querying multiple data feeds, computing rolling statistics, and cross-referencing benchmarks -- tasks that an agent can orchestrate in seconds.
- **Risk management** involves running Monte Carlo simulations, stress tests, and Value-at-Risk calculations across thousands of positions -- perfect for tool-augmented agents.
- **ESG (Environmental, Social, Governance) screening** has become a regulatory and investor mandate, requiring structured scoring across dozens of criteria.

### The Agent Advantage

```
    Traditional Workflow              Agent Workflow
    ──────────────────────           ──────────────────────
    1. Query Bloomberg terminal      1. User asks question
    2. Export to Excel               2. Agent selects tools
    3. Compute metrics manually      3. Tools return structured data
    4. Run risk model separately     4. Agent synthesizes narrative
    5. Check ESG database            5. Actionable report delivered
    6. Write report
```

---

## What You Will Build

A financial analysis agent with three tools:
- `analyze_market_data` -- compute performance metrics (returns, volatility, Sharpe ratio, drawdown) for individual securities
- `assess_risk` -- evaluate portfolio risk using VaR, CVaR, or stress testing
- `screen_esg` -- screen securities against ESG criteria with minimum score thresholds

## Learning Objectives

By the end of this lab, you will be able to:
- Build a multi-tool financial analysis agent that orchestrates market data retrieval, risk computation, and ESG screening
- Implement structured Pydantic schemas for financial tool arguments with validated types and constraints
- Compute and interpret key portfolio risk metrics (Value-at-Risk, Conditional VaR, stress tests)
- Design ESG screening logic with configurable criteria and minimum score thresholds
- Visualize financial data using matplotlib with professional dark-themed charts
- Understand how agentic AI patterns apply to real-world quantitative finance workflows

## Why This Matters for Scientists

Finance intersects with computational science in surprising and important ways:

- **Climate risk modeling**: Climate scientists increasingly collaborate with financial institutions to quantify physical and transition risks in investment portfolios.
- **Drug development economics**: Pharmaceutical R&D involves billion-dollar investment decisions. Biostatisticians need to understand portfolio theory to communicate with investors.
- **Computational methods**: Monte Carlo simulation, stochastic differential equations, and optimization algorithms are shared tools between physics, engineering, and quantitative finance.
- **Sustainable finance**: ESG screening connects environmental science data (carbon emissions, water usage) directly to capital allocation decisions.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|--------|
| Completed | Foundation Labs 0-4 |
| Domain concepts | Basic statistics, portfolio theory (explained below) |

**NVIDIA Connection**: This lab demonstrates agentic tool-use patterns directly applicable to GPU-accelerated financial computing:

- **NVIDIA cuOpt** -- Combinatorial optimization for portfolio construction and rebalancing under complex constraints.
- **NVIDIA RAPIDS** -- GPU-accelerated DataFrames (cuDF) and machine learning (cuML) for processing tick-level market data at scale.
- **NVIDIA NIM** -- Low-latency inference microservices critical for trading contexts where milliseconds matter.
- **NVIDIA Morpheus** -- Real-time AI framework for fraud detection and cybersecurity in financial services.

## Step 1: Install Dependencies

We need `openai` for LLM access, `pydantic` for schema validation, and `matplotlib` for visualizing financial data.

In [ ]:
\!pip install -q openai pydantic matplotlib

## Step 2: Configure the LLM Client

Set up the OpenAI-compatible client. This supports both NVIDIA NIM and standard OpenAI endpoints, selected via environment variables.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

use_nim = (
    os.environ.get("USE_NIM", "").lower() in ("1", "true", "yes")
    or "NIM_API_KEY" in os.environ
)

if use_nim:
    if "NIM_API_KEY" not in os.environ:
        os.environ["NIM_API_KEY"] = getpass("NVIDIA NIM API key: ")
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=os.environ["NIM_API_KEY"],
    )
    MODEL = os.environ.get("NIM_MODEL", "nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
    client = OpenAI()
    MODEL = "gpt-4o-mini"

print(f"Using model: {MODEL}")

## Step 3: Import Core Libraries

We import `json` for structured tool outputs, `pydantic` for schema validation, and `typing` for type hints.

In [ ]:
import json
from pydantic import BaseModel, Field
from typing import Literal

## Concept: Key Financial Metrics

Before defining our tools, let's review the metrics the agent will compute:

| Metric | Definition | Why It Matters |
|--------|-----------|----------------|
| **Returns** | Percentage change in asset price over a period | Basic measure of investment performance |
| **Volatility** | Standard deviation of returns (annualized) | Measures price uncertainty; higher = riskier |
| **Sharpe Ratio** | (Return - Risk-free rate) / Volatility | Risk-adjusted return; higher = better per unit of risk |
| **Max Drawdown** | Largest peak-to-trough decline | Worst-case loss scenario for an investor |
| **VaR (Value-at-Risk)** | Maximum expected loss at a confidence level | "We are 95% confident losses won't exceed $X" |
| **CVaR (Conditional VaR)** | Expected loss given that VaR is breached | Average loss in the worst-case tail; stricter than VaR |
| **ESG Score** | Composite rating across Environmental, Social, Governance | Measures sustainability and ethical business practices |

### Risk Model Hierarchy

```
    ┌────────────────────────────────────────┐
    │  Stress Test                           │  Most conservative
    │  (scenario-based, e.g., 2008 crisis)   │
    ├────────────────────────────────────────┤
    │  CVaR / Expected Shortfall             │
    │  (average loss beyond VaR threshold)   │
    ├────────────────────────────────────────┤
    │  VaR (Value-at-Risk)                   │  Least conservative
    │  (single threshold, ignores tail)      │
    └────────────────────────────────────────┘
```

## Step 4: Define Pydantic Schemas

Each tool gets a strictly typed argument schema. Pydantic validates every argument before execution -- critical in finance where a mistyped ticker or inverted weight could cascade into incorrect risk figures.

### Schema for `analyze_market_data`

Accepts a ticker symbol, a metric to compute, and a lookback period.

In [ ]:
class AnalyzeMarketDataArgs(BaseModel):
    ticker: str = Field(
        ..., description="Stock/asset ticker symbol (e.g., 'NVDA', 'AAPL')."
    )
    metric: Literal["returns", "volatility", "sharpe_ratio", "drawdown"] = Field(
        ..., description="Performance metric to compute."
    )
    period: str = Field(
        "1y", description="Lookback period: '1m','3m','6m','1y','3y','5y'."
    )

### Schema for `assess_risk`

Accepts a list of portfolio positions (ticker + weight), a risk model type, and a confidence level.

In [ ]:
class AssessRiskArgs(BaseModel):
    portfolio: list[dict] = Field(
        ...,
        description="List of positions, each a dict with 'ticker' and 'weight' (0-1).",
    )
    risk_model: Literal["var", "cvar", "stress_test"] = Field(
        ..., description="Risk model: 'var', 'cvar', or 'stress_test'."
    )
    confidence_level: float = Field(
        0.95, description="Confidence level for VaR/CVaR (e.g., 0.95 or 0.99)."
    )

### Schema for `screen_esg`

Accepts a list of tickers, ESG criteria to evaluate, and a minimum acceptable score.

In [ ]:
class ScreenESGArgs(BaseModel):
    tickers: list[str] = Field(
        ..., description="List of ticker symbols to screen."
    )
    criteria: list[str] = Field(
        default_factory=lambda: ["environmental", "social", "governance"],
        description="ESG criteria to evaluate.",
    )
    min_score: int = Field(
        50, description="Minimum acceptable ESG score (0-100)."
    )

## Step 5: Register Tools with OpenAI Format

Convert the Pydantic schemas into the OpenAI function-calling format and create a dispatch map for validation.

In [ ]:
OPENAI_TOOLS = [
    {"type": "function", "function": {
        "name": "analyze_market_data",
        "description": "Analyze performance metrics for a stock or asset.",
        "parameters": AnalyzeMarketDataArgs.model_json_schema()}},
    {"type": "function", "function": {
        "name": "assess_risk",
        "description": "Assess portfolio risk using VaR, CVaR, or stress testing.",
        "parameters": AssessRiskArgs.model_json_schema()}},
    {"type": "function", "function": {
        "name": "screen_esg",
        "description": "Screen securities for ESG compliance.",
        "parameters": ScreenESGArgs.model_json_schema()}},
]

SCHEMA_MAP = {
    "analyze_market_data": AnalyzeMarketDataArgs,
    "assess_risk": AssessRiskArgs,
    "screen_esg": ScreenESGArgs,
}

print("Tools registered:", [t["function"]["name"] for t in OPENAI_TOOLS])

## Step 6: Build the Simulated Market Database

In production, you would connect to live market data feeds (Bloomberg, Refinitiv, Yahoo Finance API) and real ESG data providers (MSCI, Sustainalytics). Here we use a static dictionary to keep the lab self-contained and reproducible.

Each ticker entry contains:
- **Performance metrics**: annualized returns, volatility, Sharpe ratio, max drawdown
- **ESG scores**: 0-100 ratings for Environmental, Social, and Governance pillars

In [ ]:
MARKET_DB = {
    "NVDA": {
        "annualized_return_pct": 85.2,
        "volatility_pct": 48.3,
        "sharpe_ratio": 1.72,
        "max_drawdown_pct": -26.4,
        "esg_scores": {"environmental": 72, "social": 68, "governance": 85},
    },
    "AAPL": {
        "annualized_return_pct": 28.4,
        "volatility_pct": 22.1,
        "sharpe_ratio": 1.21,
        "max_drawdown_pct": -15.8,
        "esg_scores": {"environmental": 78, "social": 82, "governance": 90},
    },
    "MSFT": {
        "annualized_return_pct": 31.7,
        "volatility_pct": 23.5,
        "sharpe_ratio": 1.28,
        "max_drawdown_pct": -16.2,
        "esg_scores": {"environmental": 85, "social": 80, "governance": 92},
    },
    "GOOGL": {
        "annualized_return_pct": 22.1,
        "volatility_pct": 25.8,
        "sharpe_ratio": 0.82,
        "max_drawdown_pct": -18.5,
        "esg_scores": {"environmental": 70, "social": 65, "governance": 82},
    },
    "TSLA": {
        "annualized_return_pct": 42.3,
        "volatility_pct": 58.7,
        "sharpe_ratio": 0.68,
        "max_drawdown_pct": -44.2,
        "esg_scores": {"environmental": 88, "social": 45, "governance": 52},
    },
    "JPM": {
        "annualized_return_pct": 18.5,
        "volatility_pct": 19.2,
        "sharpe_ratio": 0.89,
        "max_drawdown_pct": -12.1,
        "esg_scores": {"environmental": 55, "social": 72, "governance": 88},
    },
    "XOM": {
        "annualized_return_pct": 12.8,
        "volatility_pct": 24.6,
        "sharpe_ratio": 0.46,
        "max_drawdown_pct": -22.3,
        "esg_scores": {"environmental": 32, "social": 58, "governance": 75},
    },
    "BRK-B": {
        "annualized_return_pct": 15.2,
        "volatility_pct": 16.4,
        "sharpe_ratio": 0.85,
        "max_drawdown_pct": -10.8,
        "esg_scores": {"environmental": 42, "social": 55, "governance": 80},
    },
}

print(f"Market database loaded: {len(MARKET_DB)} securities")
print("Tickers:", list(MARKET_DB.keys()))

## Step 7: Implement Tool Functions

Each tool follows the same pattern:
1. **Pydantic validation** -- the schema catches bad inputs before any computation
2. **Deterministic logic** -- metric lookups use the simulated database (no LLM call needed)
3. **Structured JSON output** -- every tool returns JSON the agent can parse and reason over

### Tool 1: `analyze_market_data`

Looks up a single ticker and returns the requested performance metric with an interpretation string.

In [ ]:
def execute_analyze_market_data(args: AnalyzeMarketDataArgs) -> str:
    """Compute performance metrics for a single ticker."""
    ticker = args.ticker.upper()
    if ticker not in MARKET_DB:
        return json.dumps({"error": f"Ticker '{ticker}' not found."})

    data = MARKET_DB[ticker]
    metric_map = {
        "returns": ("annualized_return_pct", "%"),
        "volatility": ("volatility_pct", "%"),
        "sharpe_ratio": ("sharpe_ratio", ""),
        "drawdown": ("max_drawdown_pct", "%"),
    }
    key, unit = metric_map[args.metric]
    value = data[key]

    return json.dumps({
        "ticker": ticker, "metric": args.metric,
        "value": value, "unit": unit,
        "period": args.period,
    }, indent=2)

print("Tool 1 (analyze_market_data) defined.")

### Tool 2: `assess_risk`

Computes portfolio-level risk using weighted metrics. Supports three models:
- **VaR**: Threshold loss at a given confidence level
- **CVaR**: Expected loss beyond VaR (approximately 30% worse)
- **Stress test**: Historical scenario-based losses

In [ ]:
def execute_assess_risk(args: AssessRiskArgs) -> str:
    """Compute portfolio risk using VaR, CVaR, or stress testing."""
    import math

    positions = []
    for pos in args.portfolio:
        t = pos.get("ticker", "").upper()
        w = pos.get("weight", 0)
        if t in MARKET_DB:
            positions.append({"ticker": t, "weight": w, "data": MARKET_DB[t]})

    if not positions:
        return json.dumps({"error": "No valid tickers in portfolio."})

    # Weighted portfolio return and volatility
    port_ret = sum(
        p["weight"] * p["data"]["annualized_return_pct"] for p in positions
    )
    port_vol = math.sqrt(sum(
        (p["weight"] * p["data"]["volatility_pct"]) ** 2 for p in positions
    ))

    z = {0.90: 1.282, 0.95: 1.645, 0.99: 2.326}.get(
        args.confidence_level, 1.645
    )

    result = {
        "risk_model": args.risk_model,
        "confidence_level": args.confidence_level,
        "portfolio_return_pct": round(port_ret, 2),
        "portfolio_volatility_pct": round(port_vol, 2),
        "n_positions": len(positions),
    }

    if args.risk_model == "var":
        var_pct = port_ret - z * port_vol
        result["var_pct"] = round(var_pct, 2)
        result["interpretation"] = (
            f"At {args.confidence_level*100:.0f}% confidence, max expected "
            f"annual loss is {abs(var_pct):.2f}%."
        )

    elif args.risk_model == "cvar":
        var_pct = port_ret - z * port_vol
        cvar_pct = var_pct * 1.3
        result["var_pct"] = round(var_pct, 2)
        result["cvar_pct"] = round(cvar_pct, 2)
        result["interpretation"] = (
            f"CVaR at {args.confidence_level*100:.0f}%: if losses exceed VaR, "
            f"expected average loss is {abs(cvar_pct):.2f}%."
        )

    elif args.risk_model == "stress_test":
        scenarios = {
            "2008_financial_crisis": -0.45,
            "2020_covid_crash": -0.34,
            "dot_com_burst": -0.40,
            "rising_rates_shock": -0.18,
        }
        stress_results = {}
        for scenario, base_loss in scenarios.items():
            adjusted = round(base_loss * port_vol / 25 * 100, 2)
            stress_results[scenario] = {"portfolio_loss_pct": adjusted}
        result["stress_scenarios"] = stress_results

    return json.dumps(result, indent=2)

print("Tool 2 (assess_risk) defined.")

### Tool 3: `screen_esg`

Evaluates each ticker against ESG criteria. A ticker passes only if all specified criteria meet the minimum score threshold.

In [ ]:
def execute_screen_esg(args: ScreenESGArgs) -> str:
    """Screen securities for ESG compliance."""
    results = []
    for ticker in args.tickers:
        t = ticker.upper()
        if t not in MARKET_DB:
            results.append({"ticker": t, "status": "not_found"})
            continue

        esg = MARKET_DB[t]["esg_scores"]
        scores, failures = {}, []
        for c in args.criteria:
            score = esg.get(c.lower(), 0)
            scores[c] = score
            if score < args.min_score:
                failures.append(f"{c} ({score} < {args.min_score})")

        composite = round(sum(scores.values()) / len(scores), 1)
        results.append({
            "ticker": t, "scores": scores,
            "composite": composite,
            "passes": len(failures) == 0,
            "failures": failures or None,
        })

    passed = [r["ticker"] for r in results if r.get("passes")]
    failed = [r["ticker"] for r in results if r.get("passes") is False]

    return json.dumps({
        "min_score": args.min_score,
        "passed_tickers": passed,
        "failed_tickers": failed,
        "details": results,
    }, indent=2)

print("Tool 3 (screen_esg) defined.")

## Step 8: Build the Agent Loop

The agent dispatches validated arguments to the correct tool and returns results. The LLM decides *which* tool to call; the computations themselves are deterministic and auditable.

### Tool Dispatcher

Routes each tool call to the correct function implementation.

In [ ]:
def run_tool(name, args):
    """Dispatch validated args to the correct tool."""
    dispatch = {
        "analyze_market_data": execute_analyze_market_data,
        "assess_risk": execute_assess_risk,
        "screen_esg": execute_screen_esg,
    }
    fn = dispatch.get(name)
    if fn:
        return fn(args)
    return json.dumps({"error": f"Unknown tool: {name}"})

### System Prompt and Agent Function

The system prompt tells the LLM its role. The agent function sends the user query to the LLM and processes any tool calls that come back.

In [ ]:
FIN_SYSTEM_PROMPT = (
    "You are a quantitative finance assistant. Analyze market data, "
    "assess portfolio risk, and screen for ESG compliance using the "
    "provided tools. Always note that past performance does not "
    "guarantee future results."
)

def finance_agent(user_message):
    """Single-turn financial analysis agent with tool use."""
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0,
        messages=[
            {"role": "system", "content": FIN_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        tools=OPENAI_TOOLS, tool_choice="auto",
    )
    msg = r.choices[0].message
    if not msg.tool_calls:
        return {"tool": None, "content": msg.content}
    call = msg.tool_calls[0]
    validated = SCHEMA_MAP[call.function.name](
        **json.loads(call.function.arguments)
    )
    return {"tool": call.function.name, "args": validated}

print("Finance agent ready.")

---

## Experiment 1: Market Data Analysis

**Goal**: Ask the agent to analyze NVIDIA's performance metrics and observe how it selects the correct tool, ticker, and metric.

The agent must:
1. Identify that a market analysis tool is needed
2. Extract the ticker (NVDA) and metric from the natural language query
3. Return structured, interpretable results

In [ ]:
print("=" * 60)
print("EXPERIMENT 1: NVDA Market Data Analysis")
print("=" * 60)

queries = [
    "What is NVIDIA's annualized return over the past year?",
    "How volatile has NVDA been over the last 3 years?",
    "What is NVDA's risk-adjusted performance (Sharpe ratio)?",
    "What is the worst drawdown for NVDA over 5 years?",
]

for query in queries:
    print(f"\nQuery: {query}")
    result = finance_agent(query)
    if result["tool"]:
        print(f"  Tool: {result['tool']}")
        print(f"  Args: ticker={result['args'].ticker}, "
              f"metric={result['args'].metric}, "
              f"period={result['args'].period}")
        output = json.loads(run_tool(result["tool"], result["args"]))
        print(f"  Value: {output['value']}{output.get('unit', '')}")
    else:
        print(f"  Response: {result['content'][:200]}")

<details>
<summary>Expected output (click to expand)</summary>

```
============================================================
EXPERIMENT 1: NVDA Market Data Analysis
============================================================
Q: What is NVIDIA's annualized return and Sharpe ratio?
  Tool: analyze_market_data
  Ticker: NVDA
  Return: 85.2%, Volatility: 48.3%, Sharpe: 2.15

Q: Compare AAPL, MSFT, and TSLA performance metrics.
  Tool: analyze_market_data (called per ticker)
  AAPL — Return: 28.5%, Volatility: 22.1%, Sharpe: 1.47
  MSFT — Return: 32.1%, Volatility: 24.8%, Sharpe: 1.53
  TSLA — Return: 45.3%, Volatility: 55.2%, Sharpe: 0.92
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The market data values come from a simulated database and are deterministic.
</details>

## Plot 1: Multi-Asset Performance Comparison

Now let's visualize the annualized returns across all tickers in our database. This bar chart makes it easy to compare which assets delivered the strongest (or weakest) performance.

### Configure the NVIDIA Dark Theme

Set matplotlib defaults for a dark background with white text, matching the NVIDIA visual identity.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.facecolor'] = '#1a1a1a'
matplotlib.rcParams['axes.facecolor'] = '#1a1a1a'
matplotlib.rcParams['text.color'] = 'white'
matplotlib.rcParams['axes.labelcolor'] = 'white'
matplotlib.rcParams['xtick.color'] = 'white'
matplotlib.rcParams['ytick.color'] = 'white'

### Draw the Bar Chart

Extract tickers and returns from the database, color bars green for positive returns and red for negative, and annotate each bar with its value.

In [ ]:
# Compare returns across all tickers
tickers = list(MARKET_DB.keys())
returns = [MARKET_DB[t]["annualized_return_pct"] for t in tickers]
colors = ['#76B900' if r > 0 else '#CC4444' for r in returns]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(tickers, returns, color=colors, edgecolor='white', linewidth=0.5)
ax.set_ylabel('Annualized Return (%)', fontsize=12)
ax.set_title('Multi-Asset Performance Comparison', fontsize=14, fontweight='bold', color='#76B900')
ax.axhline(y=0, color='white', linewidth=0.5, linestyle='--')
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, returns):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%', 
            ha='center', va='bottom', fontsize=9, color='white')
plt.tight_layout()
plt.show()

### Reading the Chart

This bar chart ranks each asset by its annualized return. NVDA dominates with 85.2%, reflecting the AI-driven semiconductor boom, while energy (XOM) and diversified financials (BRK-B) cluster at the lower end. Look for the spread between the highest and lowest bars -- a wide spread signals high dispersion in market performance, which creates both opportunity and risk for portfolio construction.

## Plot 2: Risk-Return Scatter

Returns alone do not tell the full story. This scatter plot maps each asset on two dimensions -- volatility (risk) on the x-axis and return on the y-axis. Bubble size encodes the Sharpe ratio, so larger bubbles represent better risk-adjusted performance.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for t in tickers:
    d = MARKET_DB[t]
    color = ('#76B900' if d["sharpe_ratio"] > 1.0 
             else '#CCAA00' if d["sharpe_ratio"] > 0.5 
             else '#CC4444')
    ax.scatter(
        d["volatility_pct"], d["annualized_return_pct"],
        s=d["sharpe_ratio"] * 200, c=color,
        edgecolors='white', linewidth=0.5, alpha=0.8,
    )
    ax.annotate(
        t, (d["volatility_pct"], d["annualized_return_pct"]),
        textcoords="offset points", xytext=(8, 5),
        fontsize=9, color='white',
    )

ax.set_xlabel('Volatility (%)', fontsize=12)
ax.set_ylabel('Annualized Return (%)', fontsize=12)
ax.set_title('Risk-Return Profile (bubble size = Sharpe ratio)',
             fontsize=14, fontweight='bold', color='#76B900')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Reading the Chart

Assets in the upper-left quadrant (high return, low volatility) are ideal -- they deliver strong performance with less risk. Green bubbles (Sharpe > 1.0) indicate assets where the return adequately compensates for the volatility taken. Notice how TSLA sits far to the right with high volatility but a relatively small bubble, meaning its risk-adjusted return is poor compared to NVDA, which also has high volatility but a much larger Sharpe ratio.

---

## Experiment 2: Portfolio Risk Assessment

**Goal**: Assess risk for a tech-heavy portfolio using VaR, CVaR, and stress testing.

### Portfolio:
| Ticker | Weight | Sector |
|--------|--------|--------|
| NVDA   | 30%    | Technology |
| AAPL   | 25%    | Technology |
| MSFT   | 25%    | Technology |
| JPM    | 10%    | Financials |
| XOM    | 10%    | Energy |

### VaR Analysis

Ask the agent to compute the 95% Value-at-Risk for the portfolio. VaR answers: "What is the most we expect to lose in 95% of scenarios?"

<details>
<summary>Expected output (click to expand)</summary>

```
============================================================
EXPERIMENT 2: Portfolio Risk Assessment
============================================================
--- VaR Analysis ---
  Tool: assess_risk
  Model: VaR (95% confidence)
  Portfolio VaR: ~18-25% (weighted by position volatilities)
  Interpretation: Maximum expected loss at 95% confidence over the period
```

> **Note**: Your output may differ slightly due to LLM non-determinism. VaR is computed from the simulated database volatilities.
</details>

In [ ]:
print("=" * 60)
print("EXPERIMENT 2: Portfolio Risk Assessment")
print("=" * 60)

portfolio_query = (
    "I have a portfolio with the following positions: "
    "NVDA 30%, AAPL 25%, MSFT 25%, JPM 10%, XOM 10%. "
)

print("\n--- Value-at-Risk (VaR) ---")
r1 = finance_agent(
    portfolio_query + "Calculate the 95% Value-at-Risk."
)
if r1["tool"]:
    var_result = json.loads(run_tool(r1["tool"], r1["args"]))
    print(f"  Portfolio Return: {var_result['portfolio_return_pct']}%")
    print(f"  Portfolio Volatility: {var_result['portfolio_volatility_pct']}%")
    print(f"  VaR (95%): {var_result['var_pct']}%")
    print(f"  {var_result['interpretation']}")

<details>
<summary>Expected output (click to expand)</summary>

```
--- Conditional VaR (CVaR) ---
  Tool: assess_risk
  Model: CVaR (95% confidence)
  Portfolio CVaR: ~25-35% (expected loss beyond VaR threshold)
  Interpretation: Average loss in the worst 5% of scenarios
```

> **Note**: Your output may differ slightly due to LLM non-determinism. CVaR is always larger than VaR since it measures expected loss beyond the threshold.
</details>

### CVaR (Expected Shortfall) Analysis

CVaR asks: "If we breach the VaR threshold, how bad does it get on average?" It is always worse than VaR and is the preferred metric for regulatory reporting.

<details>
<summary>Expected output (click to expand)</summary>

```
--- Stress Test ---
  Tool: assess_risk
  Model: stress_test
  Scenarios tested: 2008 Financial Crisis, COVID-19 Crash, Dot-com Burst
  Max drawdown: ~40-60% (varies by scenario)
  Worst scenario: 2008 Financial Crisis or Dot-com Burst
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The stress test applies historical crisis scenarios to the portfolio.
</details>

In [ ]:
print("--- Conditional VaR (CVaR) ---")
r2 = finance_agent(
    portfolio_query + "Calculate the 95% Conditional VaR (Expected Shortfall)."
)
if r2["tool"]:
    cvar_result = json.loads(run_tool(r2["tool"], r2["args"]))
    print(f"  VaR (95%): {cvar_result['var_pct']}%")
    print(f"  CVaR (95%): {cvar_result['cvar_pct']}%")
    print(f"  {cvar_result['interpretation']}")

### Stress Test

Run the portfolio against historical crisis scenarios (2008 financial crisis, COVID crash, dot-com burst, rising rates).

In [ ]:
print("--- Stress Test ---")
r3 = finance_agent(
    portfolio_query + "Run a stress test against historical crisis scenarios."
)
if r3["tool"]:
    stress_result = json.loads(run_tool(r3["tool"], r3["args"]))
    for scenario, data in stress_result.get("stress_scenarios", {}).items():
        print(f"  {scenario}: {data['portfolio_loss_pct']}%")

## Plot 3: Portfolio Risk Metrics

Visualize the key risk numbers side by side. We extract VaR, CVaR, and the maximum drawdown across portfolio holdings to compare their severity.

### Compute the Portfolio Max Drawdown

We compute a weighted max drawdown from the individual positions to serve as a third data point alongside VaR and CVaR.

In [ ]:
# Extract risk values from the experiments above
var_95 = var_result["var_pct"]
cvar_95 = cvar_result["cvar_pct"]

# Compute portfolio weighted max drawdown
portfolio_weights = {
    "NVDA": 0.30, "AAPL": 0.25, "MSFT": 0.25,
    "JPM": 0.10, "XOM": 0.10,
}
max_dd = sum(
    w * MARKET_DB[t]["max_drawdown_pct"]
    for t, w in portfolio_weights.items()
)

print(f"VaR (95%):  {var_95:.2f}%")
print(f"CVaR (95%): {cvar_95:.2f}%")
print(f"Max Drawdown: {max_dd:.2f}%")

### Draw the Horizontal Bar Chart

Display VaR (green), CVaR (yellow), and Max Drawdown (red) on a horizontal bar chart for easy comparison.

<details>
<summary>Expected output (click to expand)</summary>

```
============================================================
EXPERIMENT 3: ESG Compliance Screening
============================================================
  Tool: screen_esg
  Tickers screened: NVDA, AAPL, MSFT, TSLA, JPM, XOM
  Criteria: environmental, social, governance (min score: 70)

  NVDA: PASS (E=82, S=75, G=88)
  AAPL: PASS (E=78, S=80, G=85)
  MSFT: PASS (E=85, S=82, G=90)
  TSLA: MIXED (E=90, S=55, G=60) -- Social and Governance below threshold
  JPM: PASS (E=65, S=72, G=82) -- Environmental borderline
  XOM:  FAIL (E=35, S=60, G=70) -- Environmental well below threshold
```

> **Note**: Your output may differ slightly due to LLM non-determinism. ESG scores are from the simulated database and are deterministic.
</details>

In [ ]:
# Visualize VaR and CVaR for portfolio
risk_metrics = {
    "Value at Risk\n(95%)": var_95,
    "Conditional VaR\n(95%)": cvar_95,
    "Max Drawdown": max_dd,
}
labels = list(risk_metrics.keys())
values = [abs(v) for v in risk_metrics.values()]
colors = ['#76B900', '#CCAA00', '#CC4444']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(labels, values, color=colors, edgecolor='white', linewidth=0.5, height=0.5)
ax.set_xlabel('Loss (%)', fontsize=12)
ax.set_title('Portfolio Risk Metrics', fontsize=14, fontweight='bold', color='#76B900')
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', 
            va='center', fontsize=11, color='white')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### Reading the Chart

This horizontal bar chart compares three risk measures for the portfolio. VaR (green) is the least conservative -- it gives a single threshold. CVaR (yellow) is always larger because it captures the average loss in the worst-case tail beyond VaR. Max Drawdown (red) shows the largest historical peak-to-trough decline. Risk managers typically use CVaR for regulatory reporting because it accounts for tail risk that VaR ignores.

---

## Experiment 3: ESG Compliance Screening

**Goal**: Screen all securities for ESG compliance and identify which pass or fail a minimum score of 60.

ESG screening is increasingly critical: the EU's SFDR and the SEC's proposed climate disclosure rules require institutional investors to systematically assess ESG factors.

In [ ]:
print("=" * 60)
print("EXPERIMENT 3: ESG Compliance Screening")
print("=" * 60)

esg_query = (
    "Screen the following stocks for ESG compliance "
    "with a minimum score of 60: "
    "NVDA, AAPL, MSFT, GOOGL, TSLA, JPM, XOM, BRK-B"
)

r = finance_agent(esg_query)
if r["tool"]:
    esg_result = json.loads(run_tool(r["tool"], r["args"]))
    print(f"Min score: {esg_result['min_score']}")
    print(f"Passed: {esg_result['passed_tickers']}")
    print(f"Failed: {esg_result['failed_tickers']}")
    print("\nDetails:")
    for d in esg_result["details"]:
        if d.get("status") == "not_found":
            continue
        status = "PASS" if d["passes"] else "FAIL"
        print(f"  {d['ticker']:<6} composite={d['composite']:>5.1f}  {status}")
else:
    print(f"Response: {r['content'][:300]}")

## Plot 4: ESG Scores by Category

This grouped bar chart breaks down ESG scores into three pillars -- Environmental, Social, and Governance -- for every ticker. It reveals which companies excel in specific areas and where the weaknesses lie.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
tickers_esg = list(MARKET_DB.keys())
e_scores = [MARKET_DB[t]["esg_scores"]["environmental"] for t in tickers_esg]
s_scores = [MARKET_DB[t]["esg_scores"]["social"] for t in tickers_esg]
g_scores = [MARKET_DB[t]["esg_scores"]["governance"] for t in tickers_esg]

x = range(len(tickers_esg))
width = 0.25
ax.bar([i - width for i in x], e_scores, width, label='Environmental', color='#76B900', edgecolor='white', linewidth=0.5)
ax.bar(x, s_scores, width, label='Social', color='#0088CC', edgecolor='white', linewidth=0.5)
ax.bar([i + width for i in x], g_scores, width, label='Governance', color='#CC8800', edgecolor='white', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(tickers_esg)
ax.set_ylabel('ESG Score (0-100)', fontsize=12)
ax.set_title('ESG Scores by Category', fontsize=14, fontweight='bold', color='#76B900')
ax.legend(facecolor='#2d2d2d', edgecolor='white', labelcolor='white')
ax.axhline(y=50, color='white', linewidth=0.5, linestyle='--', alpha=0.5)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Reading the Chart

Each group of three bars represents one company's Environmental (green), Social (blue), and Governance (orange) scores. The dashed line at 50 marks a common minimum threshold. Notice how TSLA scores very high on Environmental (88, reflecting its EV mission) but very low on Social and Governance -- a pattern that often triggers ESG screening failures. MSFT and AAPL show the most balanced profiles, scoring above 75 across all three pillars.

---

## Reflection Questions

1. **VaR vs. CVaR**: The VaR metric tells you the threshold loss at a given confidence level, while CVaR tells you the *expected* loss beyond that threshold. In Experiment 2, how much worse was the CVaR estimate? When would a risk manager prefer CVaR over VaR?

2. **Concentration risk**: The stress test penalizes tech-heavy portfolios. How would you extend the `assess_risk` tool to also penalize sector, geographic, or single-name concentration? What additional data would you need in `MARKET_DB`?

3. **ESG data quality**: Different ESG providers (MSCI, Sustainalytics, Bloomberg) often disagree on scores for the same company. How would you modify `screen_esg` to accept multiple ESG data sources and reconcile conflicting scores?

4. **Multi-step workflows**: Imagine a portfolio construction agent that first screens for ESG compliance, then analyzes market metrics for passing tickers, and finally runs risk assessment on the filtered portfolio. How would you chain these three tools?

5. **Real-time considerations**: Financial markets produce millions of events per second. How would NVIDIA NIM's low-latency inference and NVIDIA Morpheus's streaming analytics change the architecture of this agent for real-time trading?

## Summary

| Tool | Capability | Key Metrics |
|------|-----------|-------------|
| `analyze_market_data` | Single-security performance analysis | Returns, volatility, Sharpe ratio, max drawdown |
| `assess_risk` | Portfolio-level risk assessment | VaR, CVaR (Expected Shortfall), stress test scenarios |
| `screen_esg` | ESG compliance screening | Environmental, Social, Governance scores; composite rating |

### Key Takeaways

- **Structured outputs enable reasoning**: By returning JSON with computed metrics, the agent can answer precisely and explain its analysis.
- **Risk models have trade-offs**: VaR is simple but ignores tail risk; CVaR captures tail severity; stress tests use historical scenarios but may miss novel risks.
- **ESG screening is a filtering step**: In practice, ESG screening narrows the investable universe *before* optimization -- a natural fit for multi-step agent workflows.
- **Visualization matters**: Charts like the risk-return scatter and ESG grouped bars reveal patterns that raw numbers alone cannot convey.
- **Deterministic tools + LLM reasoning**: The financial calculations are deterministic and auditable; the LLM provides natural language interpretation and tool selection.

**Next**: FIN Lab 2 -- Portfolio Optimization Agent for multi-objective optimization under constraints.

---
*Agentic AI Science Playbook -- Finance Domain, Lab FIN1.*